In [38]:
from pathlib import Path
import pandas as pd

In [39]:
# Current working directory (where notebook runs)
current_path = Path().resolve()

# Go up manually depending on where you are
project_root = current_path.parent.parent   # adjust if needed

silver_file = project_root / "data" / "silver" / "BDCOM_2023_merged_cleaned.csv"

output_path = project_root / "data" / "gold"

print(silver_file)

df = pd.read_csv(silver_file)

C:\Users\annan\Desktop\Quang Dat Doc\EFREI\DataSCience\Project-urban-data-explorer\data\silver\BDCOM_2023_merged_cleaned.csv


In [40]:
df = pd.read_csv(silver_file)
df.head()

,X,Y,OBJECTID,c_ord,arro,qua,xbis,ybis,num,typ_voie,...,TYPE,Libellé TYPE (local),Code activité 47 postes,Libellé activité 47 postes,Code activité 18 postes,Libellé activité 18 postes,Code activité 8 postes,Libellé activité 8 postes),Code activité 2 postes,Libellé activité 2 postes
0,651791.0486,6.862992e+06,1,1311,1,2,651792.345590,6.862996e+06,25,RUE,...,C,Commerce,10301,Habillement,103,Equipement de la personne,3,Non Alimentaire,1,Commerce et service commercial
1,652152.0612,6.862579e+06,2,1464,1,2,652152.061200,6.862579e+06,1,RUE,...,C,Commerce,10403,Opticien,104,Santé-Beauté,3,Non Alimentaire,1,Commerce et service commercial
2,651430.1357,6.862714e+06,4,1623,1,3,651430.135700,6.862714e+06,196,RUE,...,C,Commerce,11101,Restauration traditionnelle,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial
3,651133.4910,6.862932e+06,6,2087,1,3,651130.053214,6.862939e+06,7,RUE,...,C,Commerce,10802,Soins du corps,108,Service aux particuliers,4,Service commercial,1,Commerce et service commercial
4,651124.6132,6.863066e+06,7,2157,1,3,651124.613200,6.863066e+06,20,RUE,...,C,Commerce,10502,Petit équipement du foyer,105,Equipement de la maison,3,Non Alimentaire,1,Commerce et service commercial


In [41]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace(",", "")
)

In [42]:
gold_activity = (
    df.groupby(["Libellé_activité_224_postes"])
    .agg(
        total_establishments=("OBJECTID", "count"),
        total_surface=("surf", "sum")
    )
    .reset_index()
    .sort_values(by="total_establishments", ascending=False)
)
display(gold_activity.head())

gold_activity.to_csv(
    output_path / "paris_activities_by_label.csv",
    index=False, encoding="utf-8-sig"
)

,Libellé_activité_224_postes,total_establishments,total_surface
131,Restauration rapide assise,2956,2980
23,Brasserie - Restauration continue sans tabac,2670,2689
38,Coiffure,2658,2663
125,Restaurant asiatique,1978,1981
130,Restaurant traditionnel français,1876,1908


In [43]:
gold_type = (
    df.groupby(["TYPE"])
    .agg(
        total_establishments=("OBJECTID", "count"),
        avg_surface=("surf", "mean"),
        total_surface=("surf", "sum")
    )
    .reset_index()
)
display(gold_type.head())

gold_type.to_csv(
    output_path / "paris_activities_by_type.csv",
    index=False, encoding="utf-8-sig"
)

,TYPE,total_establishments,avg_surface,total_surface
0,C,60439,1.073363,64873
1,D,38,1.000000,38
2,K,368,1.000000,368


In [44]:
gold_spatial = (
    df.groupby(["X", "Y"])
    .agg(
        nb_activities=("OBJECTID", "count"),
        total_surface=("surf", "sum")
    )
    .reset_index()
)
display(gold_spatial.head())

gold_spatial.to_csv(
    output_path / "paris_activities_by_location.csv",
    index=False, encoding="utf-8-sig"
)

,X,Y,nb_activities,total_surface
0,643570.9549,6.861661e+06,1,1
1,644258.2392,6.862815e+06,2,3
2,644443.9969,6.861145e+06,1,1
3,644714.6613,6.863664e+06,1,1
4,645002.8688,6.862871e+06,1,1


Filter only Restaurant

In [45]:
restaurant_df = df[
    df["Libellé_activité_224_postes"].str.contains(
        "restaurant|restauration|café|brasserie|bistrot",
        case=False,
        na=False
    )
].copy()

In [46]:
# We create zones by rounding coordinates
restaurant_df["zone_x"] = (restaurant_df["X"] // 1000) * 1000
restaurant_df["zone_y"] = (restaurant_df["Y"] // 1000) * 1000

restaurant_df["zone_id"] = (
    restaurant_df["zone_x"].astype(str) + "_" + restaurant_df["zone_y"].astype(str)
)
display(restaurant_df)

,X,Y,OBJECTID,c_ord,arro,qua,xbis,ybis,num,typ_voie,...,Libellé_activité_47_postes,Code_activité_18_postes,Libellé_activité_18_postes,Code_activité_8_postes,Libellé_activité_8_postes,Code_activité_2_postes,Libellé_activité_2_postes,zone_x,zone_y,zone_id
2,651430.1357,6.862714e+06,4,1623,1,3,651430.1357,6.862714e+06,196,RUE,...,Restauration traditionnelle,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,651000.0,6862000.0,651000.0_6862000.0
9,650959.5176,6.863140e+06,14,2522,1,4,650959.5176,6.863140e+06,14,RUE,...,Restauration traditionnelle,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,650000.0,6863000.0,650000.0_6863000.0
10,651045.0252,6.863252e+06,15,2556,1,4,651045.0252,6.863252e+06,12,RUE,...,Restauration traditionnelle,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,651000.0,6863000.0,651000.0_6863000.0
17,651119.5671,6.863453e+06,23,3360,2,5,651119.5671,6.863453e+06,19,RUE,...,Restauration traditionnelle,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,651000.0,6863000.0,651000.0_6863000.0
19,651239.3602,6.863224e+06,26,3551,2,5,651239.3602,6.863224e+06,38,RUE,...,Restauration traditionnelle,111,Café et Restaurant,5,Non Alimentaire,1,Commerce et service commercial,651000.0,6863000.0,651000.0_6863000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60827,654172.4449,6.864682e+06,83124,77915,19,76,654172.4449,6.864682e+06,53,AV,...,Restauration traditionnelle,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,654000.0,6864000.0,654000.0_6864000.0
60830,654843.6682,6.863208e+06,83127,80584,20,79,654843.6682,6.863208e+06,14,RUE,...,Restauration rapide,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,654000.0,6863000.0,654000.0_6863000.0
60832,656616.7118,6.862704e+06,83129,81974,20,80,656616.7118,6.862704e+06,209,BD,...,Brasserie et Restauration continue,111,Café et Restaurant,5,Restauration,1,Commerce et service commercial,656000.0,6862000.0,656000.0_6862000.0
60842,651728.5608,6.863699e+06,83147,93285,2,6,651728.5608,6.863699e+06,11,BD,...,Alimentaire spécialisé,102,Alimentaire,2,Alimentaire,1,Commerce et service commercial,651000.0,6863000.0,651000.0_6863000.0


In [47]:
gold_restaurants_by_zone = (
    restaurant_df.groupby("zone_id")
    .agg(
        nb_restaurants=("OBJECTID", "count"),
        avg_surface=("surf", "mean"),
        total_surface=("surf", "sum")
    )
    .reset_index()
    .sort_values("nb_restaurants", ascending=False)
)
display(gold_restaurants_by_zone)

,zone_id,nb_restaurants,avg_surface,total_surface
64,652000.0_6862000.0,681,1.008811,687
55,651000.0_6863000.0,678,1.019174,691
65,652000.0_6863000.0,603,1.008292,608
56,651000.0_6864000.0,550,1.000000,550
53,651000.0_6861000.0,469,1.012793,475
...,...,...,...,...
96,656000.0_6858000.0,1,1.000000,1
104,657000.0_6859000.0,1,1.000000,1
103,657000.0_6858000.0,1,1.000000,1
106,659000.0_6857000.0,1,1.000000,1


In [48]:
gold_restaurants_by_zone.to_csv(
    output_path/ "paris_restaurants_by_zone.csv",
     index=False, encoding="utf-8-sig"
)

